# Explore FVCOM GOM3 Hindcast on AWS Public Data

In [ ]:
import xarray as xr
import zarr
import intake
import xugrid as xu
import hvplot.xugrid
import panel as pn
pn.extension()

In [ ]:
intake_catalog_url = 'https://umassd-fvcom.s3.amazonaws.com/gom3/hindcast/umassd-fvcom-gom3.yml'
cat = intake.open_catalog(intake_catalog_url)

In [ ]:
list(cat)

In [ ]:
if zarr.__version__[0]=='2':
    dataset = 'UMASSD-FVCOM-GOM3-zarr2' 
elif zarr.__version__[0]=='3':
    dataset = 'UMASSD-FVCOM-GOM3-zarr3'

In [ ]:
%%time
ds = cat[dataset].to_dask()

In [ ]:
ds

In [ ]:
def add_ugrid_metadata(ds):
    """
    Assigns UGRID convention metadata to an xarray Dataset.
    """
    mesh_name = "mesh_topology"

    mesh_attrs = {
        "cf_role": "mesh_topology",
        "topology_dimension": 2,
        "node_coordinates": "lon lat",
        "face_coordinates": "lonc latc",
        "face_node_connectivity": "nv",
        "face_dimension": "nele"
    }

    if mesh_name not in ds:
        ds = ds.assign({mesh_name: xr.DataArray(0, attrs=mesh_attrs)})
    else:
        ds[mesh_name].attrs.update(mesh_attrs)

    if "nv" in ds and ds.nv.dims == ("three", "nele"):
        ds["nv"] = ds["nv"].transpose("nele", "three")

    if "nv" in ds:
        ds.nv.attrs.update({
            "cf_role": "face_node_connectivity",
            "start_index": 1
        })

    for var in ds.data_vars:
        if "node" in ds[var].dims or "nele" in ds[var].dims:
            ds[var].attrs["mesh"] = mesh_name
            ds[var].attrs["location"] = (
                "face" if "nele" in ds[var].dims else "node"
            )

    return ds

In [ ]:
ds_ugrid = add_ugrid_metadata(ds)

In [ ]:
%%time
uds = xu.UgridDataset(ds_ugrid)

## Visualize data on the triangular mesh

In [ ]:
%%time
uds['temp'].hvplot.trimesh(
    rasterize=True,
    geo=True,
    tiles='OSM',
    cmap='turbo',
    colorbar=True,
    width=600,
    height=400,
)

In [ ]:
%%time
uds['u'].hvplot.trimesh(
    rasterize=True,
    geo=True,
    tiles='OSM',
    cmap='turbo',
    colorbar=True,
    width=600,
    height=400,
)

In [ ]:
%%time
uds['h'].hvplot.trimesh(
    rasterize=True,
    geo=True,
    tiles='OSM',
    cmap='turbo',
    colorbar=True,
    width=600,
    height=400,
)